# BigQuery Advanced Lab

In [ ]:
PROJECT_ID = "project-2df9d2ad-b6f5-4ed5-997"  #@param {type:"string"}
BUCKET_NAME = "myfungcshimaanjalihsbuck"  #@param {type:"string"}
DATASET = "ecommerce"  #@param {type:"string"}
LOCATION = "US"  #@param {type:"string"}

# Real billed/gated operations -- leave False until you're at that section
RUN_BQML_STEP = True              #@param {type:"boolean"}
RUN_BQML_FORECAST_STEP = True     #@param {type:"boolean"}
RUN_VECTOR_SEARCH_STEP = True     #@param {type:"boolean"}
RUN_RLS_STEP = True               #@param {type:"boolean"}
RUN_ICEBERG_STEP = True           #@param {type:"boolean"}

def TBL(name):
    return f"`{PROJECT_ID}.{DATASET}.{name}`"


In [ ]:
!pip install --quiet google-cloud-bigquery-reservation google-cloud-datacatalog-lineage

from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
from google.cloud import storage
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

def run(sql):
    return bq_client.query(sql).result().to_dataframe()


In [ ]:
!gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
  bigqueryreservation.googleapis.com datalineage.googleapis.com dataplex.googleapis.com \
  --project={PROJECT_ID}

bq_client.create_dataset(f"{PROJECT_ID}.{DATASET}", exists_ok=True)

bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
print("dataset and bucket ready")


## 1. Native tables
`orders` has a nested `line_items` array, partitioned + clustered.

In [ ]:
run(f"""
CREATE OR REPLACE TABLE {TBL("customers")} (
  customer_id INT64, first_name STRING, last_name STRING, email STRING,
  city STRING, country STRING, region STRING, customer_segment STRING,
  signup_date DATE, churned BOOL
)
""")

run(f"""
CREATE OR REPLACE TABLE {TBL("products")} (
  product_id INT64, product_name STRING, category STRING, subcategory STRING,
  unit_price FLOAT64, description STRING
)
""")

run(f"""
CREATE OR REPLACE TABLE {TBL("orders")} (
  order_id INT64, customer_id INT64, region STRING, order_date DATE, status STRING,
  line_items ARRAY<STRUCT<product_id INT64, product_name STRING, quantity INT64, unit_price FLOAT64>>
)
PARTITION BY order_date
CLUSTER BY region, status
""")


## 2. Load the data

In [ ]:
from google.colab import files
print("Upload customers.csv, products.csv, and orders.json (select all three)")
uploaded = files.upload()


In [ ]:
def upload_to_gcs(local_filename):
    blob = bucket.blob(f"{DATASET}/{local_filename}")
    blob.upload_from_filename(local_filename)
    return f"gs://{BUCKET_NAME}/{DATASET}/{local_filename}"

customers_uri = upload_to_gcs("customers.csv")
products_uri = upload_to_gcs("products.csv")
orders_uri = upload_to_gcs("orders.json")

csv_cfg = bigquery.LoadJobConfig(source_format="CSV", skip_leading_rows=1, write_disposition="WRITE_TRUNCATE")
bq_client.load_table_from_uri(customers_uri, f"{PROJECT_ID}.{DATASET}.customers", job_config=csv_cfg).result()
bq_client.load_table_from_uri(products_uri, f"{PROJECT_ID}.{DATASET}.products", job_config=csv_cfg).result()

json_cfg = bigquery.LoadJobConfig(source_format="NEWLINE_DELIMITED_JSON", write_disposition="WRITE_TRUNCATE")
bq_client.load_table_from_uri(orders_uri, f"{PROJECT_ID}.{DATASET}.orders", job_config=json_cfg).result()
print("data loaded")


## 3. Simple queries

In [ ]:
display(run(f'SELECT customer_id, first_name, last_name, region, customer_segment FROM {TBL("customers")} LIMIT 10'))

display(run(f"""
SELECT product_name, category, unit_price FROM {TBL("products")}
WHERE category = 'Electronics' AND unit_price > 2000
ORDER BY unit_price DESC
"""))

display(run(f"""
SELECT region, COUNT(*) AS num_customers, COUNTIF(churned) AS churned_count
FROM {TBL("customers")} GROUP BY region ORDER BY num_customers DESC
"""))


## 4. Joins

In [ ]:
display(run(f"""
SELECT o.order_id, o.order_date, o.status, c.first_name, c.last_name, c.region
FROM {TBL("orders")} o JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC LIMIT 15
"""))

display(run(f"""
SELECT o.order_id, o.order_date, c.first_name, item.product_name, item.quantity, item.unit_price,
       item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
ORDER BY o.order_date DESC LIMIT 15
"""))


In [ ]:
display(run(f"""
SELECT c.region, p.category, SUM(item.quantity * item.unit_price) AS total_revenue
FROM {TBL("orders")} o, UNNEST(o.line_items) AS item
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
JOIN {TBL("products")} p ON item.product_id = p.product_id
WHERE o.status = 'Completed'
GROUP BY c.region, p.category
ORDER BY total_revenue DESC
"""))


## 5. Views
Goal: wrap the rollup join so analysts don't retype it (stores the query, not data).

In [ ]:
run(f"""
CREATE OR REPLACE VIEW {TBL("v_completed_order_details")} AS
SELECT o.order_id, o.order_date, o.region, c.customer_segment, item.product_name,
       item.quantity, item.unit_price, item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
""")

display(run(f'SELECT region, SUM(line_total) AS revenue FROM {TBL("v_completed_order_details")} GROUP BY region'))


## 6. Materialized views
Goal: same rollup, precomputed and incrementally refreshed.

In [ ]:
run(f"""
CREATE OR REPLACE MATERIALIZED VIEW {TBL("mv_region_daily_revenue")} AS
SELECT o.region, o.order_date, SUM(item.quantity * item.unit_price) AS daily_revenue,
       APPROX_COUNT_DISTINCT(o.order_id) AS num_orders
FROM {TBL("orders")} o, UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
GROUP BY o.region, o.order_date
""")

display(run(f'''
SELECT * FROM {TBL("mv_region_daily_revenue")} WHERE region = 'APAC' ORDER BY order_date DESC
'''))


## 7. Partitioning
Goal: show pruning by comparing `orders` against an unpartitioned copy.

In [ ]:
run(f'CREATE OR REPLACE TABLE {TBL("orders_unpartitioned")} AS SELECT * FROM {TBL("orders")}')


In [ ]:
print("partitioned:")
display(run(f"SELECT * FROM {TBL('orders')} WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'"))

print("unpartitioned:")
display(run(f"SELECT * FROM {TBL('orders_unpartitioned')} WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'"))


In [ ]:
display(run(f"""
SELECT * FROM {TBL("orders")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31' AND region = 'APAC' AND status = 'Completed'
"""))


## 8. External tables, BigLake & Iceberg
Goal: query data in GCS in place, then a managed Iceberg table.

In [ ]:
run(f"""
CREATE OR REPLACE EXTERNAL TABLE {TBL("orders_external")}
OPTIONS (format = 'NEWLINE_DELIMITED_JSON', uris = ['{orders_uri}'])
""")

display(run(f'SELECT order_id, order_date, status FROM {TBL("orders_external")} LIMIT 10'))


In [ ]:
import json, time

if RUN_ICEBERG_STEP:
    ICEBERG_CONNECTION = f"{PROJECT_ID}.{LOCATION}.iceberg_conn"
    !bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} --location={LOCATION} iceberg_conn

    raw = !bq show --format=prettyjson --connection {ICEBERG_CONNECTION}
    connection_sa = json.loads("\n".join(raw))["cloudResource"]["serviceAccountId"]

    !gsutil iam ch serviceAccount:{connection_sa}:roles/storage.objectAdmin gs://{BUCKET_NAME}
    time.sleep(45)

    run(f"""
    CREATE OR REPLACE TABLE {TBL("orders_iceberg")} (
      order_id INT64, customer_id INT64, region STRING, order_date DATE, status STRING
    )
    WITH CONNECTION `{ICEBERG_CONNECTION}`
    OPTIONS (
      file_format = 'PARQUET', table_format = 'ICEBERG',
      storage_uri = 'gs://{BUCKET_NAME}/{DATASET}/iceberg_warehouse/orders_iceberg'
    )
    """)

    run(f'INSERT INTO {TBL("orders_iceberg")} SELECT order_id, customer_id, region, order_date, status FROM {TBL("orders")}')

    display(run(f'SELECT region, COUNT(*) AS num_orders FROM {TBL("orders_iceberg")} GROUP BY region ORDER BY num_orders DESC'))


## 9. Pricing — on-demand vs. reservations

In [ ]:
from google.cloud import bigquery_reservation_v1

reservation_client = bigquery_reservation_v1.ReservationServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

commitments = list(reservation_client.list_capacity_commitments(parent=parent))
reservations = list(reservation_client.list_reservations(parent=parent))
print(f"capacity commitments: {len(commitments)} | reservations: {len(reservations)}")


In [ ]:
# Reference only -- real money, not run live:
print(f'''
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --capacity_commitment=true \\
#   --edition=ENTERPRISE --plan=ANNUAL --slots=100
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --reservation=true \\
#   --slots=100 --edition=ENTERPRISE my-reservation
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --reservation_assignment=true \\
#   --reservation_id=my-reservation --job_type=QUERY --assignee_type=PROJECT --assignee_id={PROJECT_ID}
''')


## 10. IAM & data access control

In [ ]:
dataset = bq_client.get_dataset(f"{PROJECT_ID}.{DATASET}")
for entry in dataset.access_entries:
    print(f"{entry.entity_type}: {entry.entity_id} -> {entry.role or '(implied)'}")


In [ ]:
STUDENT_EMAIL = "himanshu.rathi@talktotechie.com"  # <-- set to a real user/group before uncommenting

table_ref = bq_client.get_table(f"{PROJECT_ID}.{DATASET}.v_completed_order_details")
policy = bq_client.get_iam_policy(table_ref)
policy.bindings.append({"role": "roles/bigquery.dataViewer", "members": {f"user:{STUDENT_EMAIL}"}})
bq_client.set_iam_policy(table_ref, policy)  # uncomment once STUDENT_EMAIL is real


## 11. BigQuery ML
Goal: four questions, one pattern — train, evaluate, predict.

In [ ]:
if RUN_BQML_STEP:
    run(f"""
    CREATE OR REPLACE MODEL {TBL("churn_model")}
    OPTIONS(model_type='logistic_reg', input_label_cols=['churned']) AS
    SELECT c.customer_segment, c.region, COUNT(o.order_id) AS num_orders,
           IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value, c.churned
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.churned, c.customer_id
    """)

    display(run(f'SELECT * FROM ML.EVALUATE(MODEL {TBL("churn_model")})'))


In [ ]:
if RUN_BQML_STEP:
    run(f"""
    CREATE OR REPLACE MODEL {TBL("ltv_model")}
    OPTIONS(model_type='linear_reg', input_label_cols=['lifetime_value']) AS
    SELECT c.customer_segment, c.region,
           DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days,
           IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.signup_date, c.customer_id
    """)

    display(run(f'SELECT * FROM ML.EVALUATE(MODEL {TBL("ltv_model")})'))

    display(run(f"""
    SELECT customer_segment, region, tenure_days, predicted_lifetime_value
    FROM ML.PREDICT(MODEL {TBL("ltv_model")}, (
      SELECT c.customer_segment, c.region, DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
      FROM {TBL("customers")} c LIMIT 10
    ))
    """))


In [ ]:
if RUN_BQML_STEP:
    run(f"""
    CREATE OR REPLACE MODEL {TBL("customer_segments_model")}
    OPTIONS(model_type='kmeans', num_clusters=3) AS
    SELECT COUNT(o.order_id) AS num_orders,
           IFNULL(SUM(item.quantity * item.unit_price), 0) AS total_spend,
           DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_id, c.signup_date
    """)

    display(run(f'SELECT centroid_id, feature, numerical_value FROM ML.CENTROIDS(MODEL {TBL("customer_segments_model")}) ORDER BY centroid_id, feature'))

    display(run(f"""
    SELECT c.customer_id, c.customer_segment, CENTROID_ID
    FROM ML.PREDICT(MODEL {TBL("customer_segments_model")}, (
      SELECT c.customer_id, COUNT(o.order_id) AS num_orders,
             IFNULL(SUM(item.quantity * item.unit_price), 0) AS total_spend,
             DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
      FROM {TBL("customers")} c
      LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
      LEFT JOIN UNNEST(o.line_items) AS item
      GROUP BY c.customer_id, c.signup_date
    )) AS predicted
    JOIN {TBL("customers")} c ON predicted.customer_id = c.customer_id
    ORDER BY CENTROID_ID
    """))


In [ ]:
if RUN_BQML_FORECAST_STEP:
    run(f"""
    CREATE OR REPLACE MODEL {TBL("revenue_forecast_model")}
    OPTIONS(model_type='ARIMA_PLUS', time_series_timestamp_col='order_date',
            time_series_data_col='daily_revenue', auto_arima=TRUE) AS
    SELECT o.order_date, SUM(item.quantity * item.unit_price) AS daily_revenue
    FROM {TBL("orders")} o, UNNEST(o.line_items) AS item
    WHERE o.status = 'Completed'
    GROUP BY o.order_date
    """)

    display(run(f'SELECT * FROM ML.ARIMA_EVALUATE(MODEL {TBL("revenue_forecast_model")})'))

    display(run(f"""
    SELECT forecast_timestamp, forecast_value, prediction_interval_lower_bound, prediction_interval_upper_bound
    FROM ML.FORECAST(MODEL {TBL("revenue_forecast_model")}, STRUCT(14 AS horizon, 0.95 AS confidence_level))
    ORDER BY forecast_timestamp
    """))


## 12. Time travel & change history

In [ ]:
display(run(f"SELECT order_id, status FROM {TBL('orders')} WHERE order_id = 1001"))
run(f"UPDATE {TBL('orders')} SET status = 'Cancelled' WHERE order_id = 1001")
display(run(f"SELECT order_id, status FROM {TBL('orders')} WHERE order_id = 1001"))

display(run(f"""
SELECT order_id, status FROM {TBL("orders")}
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 5 MINUTE)
WHERE order_id = 1001
"""))


In [ ]:
import time
from google.cloud.bigquery import TableReference

run(f'CREATE OR REPLACE TABLE {TBL("scratch_table")} AS SELECT * FROM {TBL("products")} LIMIT 5')
time.sleep(10)
snapshot_ms = int(time.time() * 1000)

run(f'DROP TABLE {TBL("scratch_table")}')

source_ref = TableReference.from_string(f"{PROJECT_ID}.{DATASET}.scratch_table@{snapshot_ms}")
dest_ref = TableReference.from_string(f"{PROJECT_ID}.{DATASET}.scratch_table")
bq_client.copy_table(source_ref, dest_ref).result()

display(run(f'SELECT * FROM {TBL("scratch_table")}'))


## 13. Vector Search for RAG

In [ ]:
import json, time

if RUN_VECTOR_SEARCH_STEP:
    !gcloud services enable aiplatform.googleapis.com bigqueryconnection.googleapis.com --project={PROJECT_ID} --quiet

    CONNECTION_ID = f"{PROJECT_ID}.{LOCATION}.embedding_conn"
    !bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} --location={LOCATION} embedding_conn

    raw = !bq show --format=prettyjson --connection {CONNECTION_ID}
    embedding_sa = json.loads("\n".join(raw))["cloudResource"]["serviceAccountId"]
    time.sleep(30)
    !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{embedding_sa}" --role="roles/owner" --quiet --condition=None
    time.sleep(30)

    run(f"""
    CREATE OR REPLACE MODEL {TBL("embedding_model")}
    REMOTE WITH CONNECTION `{CONNECTION_ID}`
    OPTIONS (endpoint = 'text-embedding-004')
    """)


In [ ]:
if RUN_VECTOR_SEARCH_STEP:
    run(f"""
    CREATE OR REPLACE TABLE {TBL("product_embeddings")} AS
    SELECT product_id, product_name, ml_generate_embedding_result AS embedding
    FROM ML.GENERATE_EMBEDDING(
      MODEL {TBL("embedding_model")},
      (SELECT product_id, product_name, description AS content FROM {TBL("products")})
    )
    """)

    display(run(f"""
    SELECT base.product_name, distance
    FROM VECTOR_SEARCH(
      TABLE {TBL("product_embeddings")}, 'embedding',
      (SELECT embedding FROM {TBL("product_embeddings")} WHERE product_name = 'Wireless Mouse'),
      top_k => 5
    )
    """))


## 14. Remote functions & connections
Reference only — needs a real deployed endpoint.

In [ ]:
print(f'''
-- 1. bq mk --connection --connection_type=CLOUD_RESOURCE --location={LOCATION} remote_conn
-- 2. grant the connection's service account Cloud Functions Invoker on your endpoint

CREATE OR REPLACE FUNCTION {TBL("convert_to_usd")}(amount FLOAT64, currency STRING)
RETURNS FLOAT64
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.remote_conn`
OPTIONS (endpoint = 'https://YOUR_CLOUD_FUNCTION_URL')
''')


## 15. Data lineage

In [ ]:
from google.cloud import datacatalog_lineage_v1

lineage_client = datacatalog_lineage_v1.LineageClient()
target = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET}/tables/v_completed_order_details"

request = datacatalog_lineage_v1.SearchLinksRequest(
    parent=f"projects/{PROJECT_ID}/locations/{LOCATION.lower()}",
    target=datacatalog_lineage_v1.EntityReference(fully_qualified_name=target),
)
for link in lineage_client.search_links(request=request):
    print(f"{link.source.fully_qualified_name}  ->  {link.target.fully_qualified_name}")


## 16. Row-level security

In [ ]:
import google.auth, google.auth.transport.requests, requests

credentials, _ = google.auth.default()
credentials.refresh(google.auth.transport.requests.Request())
me = requests.get("https://www.googleapis.com/oauth2/v1/userinfo",
                   headers={"Authorization": f"Bearer {credentials.token}"}).json()["email"]

if RUN_RLS_STEP:
    run(f"""
    CREATE OR REPLACE ROW ACCESS POLICY apac_only
    ON {TBL("orders")}
    GRANT TO ('user:{me}')
    FILTER USING (region = 'APAC')
    """)


## 17. Encryption — AEAD & dynamic data masking

In [ ]:
display(run(f"""
DECLARE keyset BYTES DEFAULT KEYS.NEW_KEYSET('AEAD_AES_GCM_256');
SELECT customer_id, AEAD.ENCRYPT(keyset, email, CAST(customer_id AS STRING)) AS encrypted_email
FROM {TBL("customers")} LIMIT 10
"""))


In [ ]:
print(f'''
ALTER TABLE {TBL("customers")}
ALTER COLUMN email
SET OPTIONS (policy_tags = ['projects/{PROJECT_ID}/locations/{LOCATION}/taxonomies/TAXONOMY_ID/policyTags/POLICY_TAG_ID']);
''')


## Cleanup

In [ ]:
CONFIRM_DELETE = True  #@param {type:"boolean"}

if CONFIRM_DELETE:
    bq_client.delete_dataset(f"{PROJECT_ID}.{DATASET}", delete_contents=True, not_found_ok=True)
    bucket.delete(force=True)
    for conn in ["iceberg_conn", "embedding_conn", "remote_conn"]:
        !bq rm --connection --project_id={PROJECT_ID} --location={LOCATION} -f {conn}
    print("cleanup complete")
